In [ ]:
import tkinter as tk
from tkinter import messagebox
import json
import difflib

FILE_NAME = "data.json"

# Load data
def load_data():
    try:
        with open(FILE_NAME, "r") as file:
            return json.load(file)
    except:
        return {"lost": [], "found": []}

# Save data
def save_data(data):
    with open(FILE_NAME, "w") as file:
        json.dump(data, file, indent=4)

# Add item
def add_item(item_type):
    name = entry_name.get()
    color = entry_color.get()
    location = entry_location.get()
    contact = entry_contact.get()

    if name == "" or contact == "":
        messagebox.showerror("Error", "Name and Contact required!")
        return

    data = load_data()

    item = {
        "name": name,
        "color": color,
        "location": location,
        "contact": contact
    }

    data[item_type].append(item)
    save_data(data)

    messagebox.showinfo("Success", f"{item_type.capitalize()} item added!")

    check_match(item, item_type)

# Matching
def check_match(new_item, item_type):
    data = load_data()
    opposite = "found" if item_type == "lost" else "lost"

    for item in data[opposite]:
        similarity = difflib.SequenceMatcher(None, new_item["name"], item["name"]).ratio()
        if similarity > 0.6:
            messagebox.showinfo("Match Found!", f"Contact: {item['contact']}")

# View all items
def view_items():
    data = load_data()
    text = ""

    for t in ["lost", "found"]:
        text += f"\n--- {t.upper()} ITEMS ---\n"
        for i, item in enumerate(data[t]):
            text += f"{i}: {item['name']} ({item['color']}) - {item['location']} | {item['contact']}\n"

    messagebox.showinfo("All Items", text if text else "No items found")

# Search item
def search_item():
    keyword = entry_name.get()
    data = load_data()
    result = ""

    for t in ["lost", "found"]:
        for item in data[t]:
            similarity = difflib.SequenceMatcher(None, keyword, item["name"]).ratio()
            if similarity > 0.5:
                result += f"{item['name']} ({item['color']}) - {item['location']} | {item['contact']}\n"

    messagebox.showinfo("Search Result", result if result else "No match found")

# Delete item
def delete_item():
    index = entry_delete.get()
    item_type = var_type.get()

    if index == "":
        messagebox.showerror("Error", "Enter index")
        return

    data = load_data()

    try:
        index = int(index)
        removed = data[item_type].pop(index)
        save_data(data)
        messagebox.showinfo("Deleted", f"{removed['name']} removed")
    except:
        messagebox.showerror("Error", "Invalid index")

# GUI
root = tk.Tk()
root.title("Smart Lost & Found System")
root.geometry("450x450")

tk.Label(root, text="Item Name").pack()
entry_name = tk.Entry(root)
entry_name.pack()

tk.Label(root, text="Color").pack()
entry_color = tk.Entry(root)
entry_color.pack()

tk.Label(root, text="Location").pack()
entry_location = tk.Entry(root)
entry_location.pack()

tk.Label(root, text="Contact").pack()
entry_contact = tk.Entry(root)
entry_contact.pack()

tk.Button(root, text="Add Lost Item", command=lambda: add_item("lost")).pack(pady=5)
tk.Button(root, text="Add Found Item", command=lambda: add_item("found")).pack(pady=5)

tk.Button(root, text="View All Items", command=view_items).pack(pady=5)
tk.Button(root, text="Search Item", command=search_item).pack(pady=5)

# Delete section
tk.Label(root, text="Delete Item Index").pack()
entry_delete = tk.Entry(root)
entry_delete.pack()

var_type = tk.StringVar(value="lost")
tk.Radiobutton(root, text="Lost", variable=var_type, value="lost").pack()
tk.Radiobutton(root, text="Found", variable=var_type, value="found").pack()

tk.Button(root, text="Delete Item", command=delete_item).pack(pady=5)

root.mainloop()